In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-nano",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import numpy as np

In [ ]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_API_KEY'])
client = OpenAI()
response = client.responses.create(
    model='gpt-5-nano',
    input='이번 2025년도 크리스마스에, 잠실에 열리는 크리스마스 이벤트에 대해서 설명해줘'
)

print(response.output_text)

In [ ]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_API_KEY'])
client = OpenAI()
def ask_llm(prompt, model='gpt-5-nano',temp=1):
    response = client.chat.completions.create(
        model=model,
        temperature=temp,
        messages = [
            {"role": "user", "content": prompt}
                    ]
    )
    return response.choices[0].message.content
    
# temperature / temp
# temperature는 창의성(무작위성)을 조절하는 파라미터야.
# 의미 :
# 0에 가까울수록: 정답이 고정됨, 가장 논리적·일관된 답을 선택
# 높을수록: 다양하고 창의적인 답변을 생성


# completions.create()
# client.chat.completions.create()는 채팅 방식의 모델 응답을 생성하는 함수야.

# 역할 :
# 프롬프트(사용자 메시지)를 모델에게 전달하고
# 모델이 생성한 텍스트를 반환함

# 핵심 :
# 예전에는 openai.ChatCompletion.create()였지만
# 새로운 SDK에서는 client.chat.completions.create() 형태로 바뀜

In [ ]:
# 1. zero-shot prompt
# 에시없이 지시사항만 던지는 것 - LLM 의 기본기능
prompt = "이 문장의 감정을 분류해 : '오늘의 점심 메뉴가 품절되어서 너무 슬퍼.'"
print(ask_llm(prompt,temp=1))

In [ ]:
# 2. few-show Prompting
# 이렇게 하는 거야 라고 예시(shot)를 몇개 보여줘서 성능을 높이는 기술
prompt = """
단어를 이모지로 바꿔줘
사과 -> 🍎
자동차 -> 🚗
고양이 -> 🐱
비행기->
"""
print(ask_llm(prompt,temp=1))

In [ ]:
# 3. Chain-of-Thought Prompting Cot(생각의 사슬)
prompt = """
질문 : 5개의 사과중 2개를 먹고 3개를 더 샀어.
사과의 단가는 100원이고
총 지출금액은 800원
총 남은 사과의 개수와 이사람이 사과를 구입해서 사용한 비용을 예상해줘
"""
print(ask_llm(prompt,temp=1))

In [ ]:
# 4. self-Consistnecy(자기일관성) ⭐
    # 원하는 문맥이 나올때까지 반복 돌리기? 그리고 적절한것을 사용자가 선택
    # 혹은 검토용 모델이 선정
# 한번만 묻지 않고 여러번(예 : 3번) 물어본 뒤 가장 많이 나온 답을 채택한다.
question = '철수는 학교까지 10분 걸려, 왕복은 몇분걸릴까?'
answer = []

for _ in range(3):
    answer.append(ask_llm(question))
print(f'수집된 답변들 : ', answer)
from collections import Counter
counter = Counter(answer)
counter

In [ ]:
# 5. Generate Knowlege Prompting(지식생성)
# 바로 답하지말고 관련된 지식을 먼저 생성하 뒤에 그 지식을 바탕으로 답하게 한다.
# 1단계 : 지식생성
knowledge = ask_llm('골프라는 스포츠에 대해 사실적인 지식 3가지만 나열해줘')
# 2단계 : 지식을 활용해 답변
prompt = f"""
다음 지식을 참고해서 '골프에서 홀인원이 왜 어려운지' 설명해줘.
[지식] : {knowledge}
모든 답변은 한글 혹은 영어로 작성
"""
print(ask_llm(prompt))

In [ ]:
# 6. Prompt Chaining(프롬프트 체이닝)
# 복잡한 일을 한번에 시키지 않고 A작업의 결과를 B작업의 입력으로
# 순차적으로 넘겨주는 파이프라인
# Step 1 : 주제 추출
text = '이메일 : 안녕하세요, 이번주 금요일 회의는 2시로 변경되었습니다.'
topic = ask_llm(f'다음 텍스트에서 핵심 주제만 단어로 뽑아줘 출력은 한글로 : {text}')

# Step 2 : 답장 작성
reply = ask_llm(f"'{topic}'에 대해 '알겠습니다'라는 정중한 답장 메일을 써줘")
print(reply)

In [ ]:
##

In [ ]:
# 8. Retrieval Augmented Generation(RAG 검색 증강 생성)
# 이론 : LLM 이 모르는 외부데이터(회사문서등)을 찾아서 (Retrieval)프롬프트에 넣어주고 답하게 한다.

# 가상의 검색된 문서
retrieved_doc = "문서내용 : 우리회사의 재책 근무는 매주 수요일 가능하다."

prompt = f'''
아래[참조문서]를 기반으로 답변해, 문서에 없으며 모른다고 해.
[참조문서] : {retrieved_doc}
질문 : 재택근부는 언제 할 수 있어?
'''

print(ask_llm(prompt))

In [ ]:
# 9 Automaric Reasoning and Tool-use 자동 추론 및 도구 사용
# LLM이 스스로 계산기나 검색엔진 같은 도구가 필요한지 판단하고 호출형식을 뱉어내는 것
prompt = '''
계산이 필요하면  [CALC: 수식] 이라고 출력해.
질문  : 3452 * 192 는 뭐야?
'''
# 실제 내부적으로 파이썬 코드로 계산한다.
response = ask_llm(prompt)
print(response)

In [ ]:
prompt = """
너는 자동 도구 선택 시스템이야.
다음과 같은 도구를 사용할 수 있어.

1. 계산기 -> [CALC:수식]
2. 날씨 조회 -> [WEATHER:도시명]
3. 일반질문 ->  직접입력

규칙 : 
계산이 필요하면  [CALC:...] 출력
날씨정보가 필요하면 [WEATHER:도시명]
그외는 일반적인 답변

질문 :
서울의 내일 날씨는 어때?
"""

import re
def process_response(text):
    # 계산기
    calc = re.findall(r'\[CALC:\s*(.*?)\]',text)
    if calc:
        expr = calc[0]
        return eval(expr)
    # 날씨
    weather = re.findall(r'\[WEATHER:\s*(.*?)\]',text)
    if weather:
        city = weather[0]
        return 'NotImplementedError'
    
response = ask_llm(prompt)
print(response)

for res in response.split('\n'):
    process_response(res)
##

[WEATHER:서울]


In [31]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ['OPEN_WEATHER_KEY']
lat = 37.25
lon = 126.45
url = f'https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}'
import requests
response = requests.get(url)
response

<Response [401]>

In [34]:
# 10. Automatic Prompt Engineer (APE)
# 사람이 프롬프트는 짜는 게 아니라 LLM에게 좋은 프로프트를 써줘 라고 시키는 것
task = '고객의 리뷰에서 감정을 분석하는 작업'
prompt = f'''
나는  이{task}을 하려고해.
이작업을 수행하기에 가장 환벽한 프롬프트 지시문을 간결하게 작성해줘
'''

best_prompt = ask_llm(prompt)
print(best_prompt)

다음 고객 리뷰를 읽고 감정을 분석해 JSON 형식으로 응답하시오.

- overall_sentiment: "positive" | "negative" | "neutral" 중 하나
- intensity: 0 ~ 5의 정수(감정의 강도)
- key_factors: 감정의 주요 원인으로 꼽을 수 있는 최대 3개의 키워드 배열
- reason: 감정의 근거를 한 문장으로 간략 요약
- improvement: 개선 포인트가 있으면 한 문장으로 제안(없으면 비워도 됨)

출력 예시:
{
  "overall_sentiment": "positive",
  "intensity": 4,
  "key_factors": ["가격", "품질"],
  "reason": "가성비가 좋고 품질에 만족했기 때문입니다.",
  "improvement": "배송 시간을 더 단축하면 좋겠습니다."
}


In [ ]:
# 11. Active-Prompt
# LLM이 답변하기 애매하거나 불확실한 문제를 찾아내서 사람에게 이것좀 가르쳐 주세요(예시추가)라고
# 요청하는 방식
# LLM에게 문제를 풀게 하고 '확신도(Confidence socre)를 묻는다 낮으면 그 문제를 few-shot에 예제로 추가